In [2]:
import pandas as pd
import configure_lichess_csv as cfg
import chess
import chess.engine
from stockfish import Stockfish
import dask.dataframe as dd
import swifter

Sample notebook for selecting 5000 rows from the LiChess database. 

In [70]:
df_5000 = pd.read_csv("lichess_db_puzzle.csv.zst", 
                 compression="zstd", # we use this argument to open a .zst file with no errors 
                 nrows=5000, 
                 skiprows=range(0, 1000000), # we are selecting from the 1000000th row of the data 
                 header=None)

In [71]:
df_5000.iloc[:, 2]

0                                     d1d2 b3f3
1                 f5h5 d7f6 h5e5 e7e5 d4e5 f6d5
2                           d8h4 g4d1 g2f1 d1d4
3                           g7g6 h4h5 g6h5 g4h5
4                                     f3e5 e2h2
                         ...                   
4995              e5e4 e3h6 h7g8 f6g6 f7g6 f2f8
4996                        c4b3 f2f1 a1f1 f8f1
4997                        b2d4 e4b1 g1f2 b1h1
4998                        b5c3 g7c3 c8b7 a1h1
4999    a2c3 f6c3 b4b5 c6d6 b5b6 c3f6 b6b7 d6c7
Name: 2, Length: 5000, dtype: object

In [85]:
df_5000["LegalMoves"] = df_5000.iloc[:, 1].apply(cfg.get_all_legal_moves)

Convert legal moves into a list format. 

In [87]:
df_5000["input"] = (
    "FEN: " + df_5000.iloc[:, 1].astype(str) +
    "\nLegal Moves: " + df_5000["LegalMoves"].apply(lambda x: ", ".join(x))
)

In [88]:
df_5000["input"].iloc[1]

'FEN: r1b2r1k/1p1nq1pp/p7/P2BpQ2/3P4/5N2/1P3PPP/2R2RK1 w - - 1 20\nLegal Moves: f5f8, f5h7, f5f7, f5d7, f5g6, f5f6, f5e6, f5h5, f5g5, f5e5, f5g4, f5f4, f5e4, f5h3, f5d3, f5c2, f5b1, d5g8, d5f7, d5b7, d5e6, d5c6, d5e4, d5c4, d5b3, d5a2, f3g5, f3e5, f3h4, f3d2, f3e1, g1h1, f1e1, f1d1, c1c8, c1c7, c1c6, c1c5, c1c4, c1c3, c1c2, c1e1, c1d1, c1b1, c1a1, d4e5, h2h3, g2g3, b2b3, h2h4, g2g4, b2b4'

In [24]:
df_5000["input"]

0       FEN: 1R6/7p/5k2/5p2/2P1pK2/1r6/3b3P/3R4 w - - ...
1       FEN: r1b2r1k/1p1nq1pp/p7/P2BpQ2/3P4/5N2/1P3PPP...
2       FEN: 3Q1nk1/5p2/6r1/8/2PPP1qp/2R3P1/P5BP/6K1 w...
3           FEN: 8/2k2pp1/p7/1pK5/1P4PP/8/8/8 b - - 10 56
4       FEN: 8/1k6/8/1P1R2B1/2pP3p/2P2Nbb/4r3/6RK w - ...
                              ...                        
4995    FEN: 5r2/r4p1k/5R1p/1ppqp3/p5bP/P2PQ3/1PP2RP1/...
4996    FEN: r4r2/2p3kp/p1n3p1/1p6/2BP4/P1Q1R3/1P3qPP/...
4997    FEN: 6k1/3R1pp1/p3p1bp/1p2P1b1/1P2q3/P3P3/1B1Q...
4998    FEN: 2k4r/p2r2Qp/1p2p3/1n3pN1/8/4P3/2P1K2P/R6q...
4999     FEN: 8/6p1/2k1Pb1p/5P2/1P4P1/5K2/N7/8 w - - 2 57
Name: input, Length: 5000, dtype: object

In [79]:
# Create a df column for the instruction in the model-response pairs 
df_5000["instruction"] = "Select the best move for this position. Please justify your reasoning in 1 or 2 sentences."

In [89]:
df_5000["BestMove"] = df_5000.iloc[:, 1].apply(cfg.get_best_move)

In [90]:
df_5000["output"] = df_5000["BestMove"].apply(cfg.format_output)

In [83]:
df_5000[["instruction", "input", "output"]].to_json(
    "chess_data_5000.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)